In [1]:
#QUESTO SCRIPT TOGLIE IL PRIMO MINUTO DI DATI X MUNGITURA, IPOTIZZANDO CHE LE PRIME LETTURE
#NON SI RIFERISCANO ALL ANIMALE PRESENTE MA A QUELLO PRECEDENTE. DOPODICHE FA UN ULTERIORE TAGLIO 
#ANDANDO A PRENDERE I SUCCESSIVI 240 SEC (4MIN) PER MUNGITURA

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import os

In [2]:
path="/mnt/c/Users/isabe/OneDrive/Desktop/Lavoro/Tuscia/chiara/peaks_background/bufale_isa/"
os.chdir(path)

In [3]:
mandria_path = './dati_bufale/dati_grezzi_agosto_luglio/Dataset_CH4/Bufale_Rinna/Luglio/protocollo_2/ch4_clean_out_neg_bg_10.csv'
mandria = pd.read_csv(mandria_path)

In [4]:
mandria['ID_animal'] = mandria['ID_animal'].apply(lambda x: int(x))
mandria['Milking_number'] = mandria['Milking_number'].apply(lambda x: int(x))

In [5]:
mandria.head()

,timestamp,CH4,date,Ora inizio,Produzione (kg),ID_animal,Milking_duration,Milking_number
0,2024-07-01 09:54:00.472160,102.1,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,1
1,2024-07-01 09:54:01.474686,92.2,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,1
2,2024-07-01 09:54:02.476053,131.4,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,1
3,2024-07-01 09:54:03.478415,128.6,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,1
4,2024-07-01 09:54:04.481498,34.3,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,1


In [6]:
#dividing and plotting each milking (keeping animal information)
#inspection of milking with calculus of mean, median, standard deviation without removal of data

animals = list(set(mandria['ID_animal']))
mungiture = list(set(mandria['Milking_number']))
print(len(animals))
print(len(mungiture))

matrix= list()

mandria_cutted=mandria

36
1532


In [8]:
mandria_cutted.tail()

,timestamp,CH4,date,Ora inizio,Produzione (kg),ID_animal,Milking_duration,Milking_number
378932,2024-07-23 12:41:09.669489,37.53,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,1740
378933,2024-07-23 12:41:10.671814,62.63,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,1740
378934,2024-07-23 12:41:11.673571,15.73,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,1740
378935,2024-07-23 12:41:12.675888,15.63,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,1740
378936,2024-07-23 12:41:13.678366,19.33,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,1740


In [9]:
mandria_cutted['Milking_duration'] = mandria_cutted['Milking_duration'].apply(lambda x: str(x))


In [10]:
# Convert 'milking_duration' to seconds
mandria_cutted['milking_duration_seconds'] = mandria_cutted['Milking_duration'].apply(lambda x: int(x.split('.')[0]) * 60 + int(x.split('.')[1]))

#Assign a progressive number of seconds starting from 1 within each milking session
mandria_cutted['progressive_seconds'] = mandria_cutted.groupby(['ID_animal', 'Milking_number']).cumcount() + 1

# Filter the DataFrame to keep only records after the first minute (60 seconds)
mandria_cutted = mandria_cutted[mandria_cutted['milking_duration_seconds'] >= 60]

print(mandria_cutted.shape)
print(len(set(mandria_cutted['Milking_number'])))

# Filter the DataFrame to keep only records of milkings longer than 200 seconds
filtered_df = mandria_cutted[mandria_cutted['milking_duration_seconds'] >= 240]

print(filtered_df.shape)
print(len(set(filtered_df['Milking_number'])))


(378937, 10)
1532
(378916, 10)
1531


In [11]:
#taglio a 4 minuti per mungitura (240+60=300)
filtered_df2 = filtered_df[filtered_df['progressive_seconds'] <= 300]

In [12]:
filtered_df2.shape

(266668, 10)

In [13]:
print(len(set(filtered_df2['Milking_number'])))


1531


In [14]:
ch4_mean = filtered_df2['CH4'].mean()
print(ch4_mean)

# Salva il DataFrame combinato in un nuovo file CSV.
#mandria_path = './dati_bufale/dati_grezzi_agosto_luglio/Dataset_CH4/Bufale_Rinna/Luglio/ch4_clean_out_bg_10.csv'

output_file = './dati_bufale/dati_grezzi_agosto_luglio/Dataset_CH4/Bufale_Rinna/Luglio/protocollo_2/ch4_cutted_milkings.csv'
filtered_df2.to_csv(output_file, index=False)




108.94608507612226


In [15]:
ch4_cutted_milkings = './dati_bufale/dati_grezzi_agosto_luglio/Dataset_CH4/Bufale_Rinna/Luglio/protocollo_2/ch4_cutted_milkings.csv'
ch4_cut_milkings = pd.read_csv(ch4_cutted_milkings)
ch4_cut_milkings.head()

,timestamp,CH4,date,Ora inizio,Produzione (kg),ID_animal,Milking_duration,Milking_number,milking_duration_seconds,progressive_seconds
0,2024-07-01 09:54:00.472160,102.1,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,1,567,1
1,2024-07-01 09:54:01.474686,92.2,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,1,567,2
2,2024-07-01 09:54:02.476053,131.4,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,1,567,3
3,2024-07-01 09:54:03.478415,128.6,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,1,567,4
4,2024-07-01 09:54:04.481498,34.3,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,1,567,5


In [16]:
ch4_cut_milkings.tail()

,timestamp,CH4,date,Ora inizio,Produzione (kg),ID_animal,Milking_duration,Milking_number,milking_duration_seconds,progressive_seconds
266663,2024-07-23 12:41:09.669489,37.53,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,1740,273,9
266664,2024-07-23 12:41:10.671814,62.63,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,1740,273,10
266665,2024-07-23 12:41:11.673571,15.73,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,1740,273,11
266666,2024-07-23 12:41:12.675888,15.63,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,1740,273,12
266667,2024-07-23 12:41:13.678366,19.33,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,1740,273,13
